# APH/PPH Split — Temporal Ordering & Denominators (MIC-7286)

Blind verification traces for the APH/PPH temporal-ordering iteration. Each section maps to a plan expectation:

1. **APH incidence risk** is a probability in [0, 1] and is applied only to still/live births (0 for `partial_term`).
2. **APH per-birth > per-pregnancy** — dropping the abortion/miscarriage (c995) + ectopic (c374) terms from the denominator raises APH risk (directional artifact trace).
3. **Intrapartum-disorder incidence risks** use denominator `birth_rate - aph_csmr`, so each is slightly larger than `incidence / birth_rate`.
4. **Two-pass mortality / APH-survival gating**: APH deaths and intrapartum-disorder deaths are disjoint; APH-dead simulants carry no intrapartum disorder.
5. **AME** only on `partial_term`; APH/PPH/intrapartum only on still/live births.
6. Only **severe** hemorrhage cases die of hemorrhage.

In [ ]:
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import vivarium_gates_mncnh
from vivarium import Artifact, InteractiveContext
from vivarium.framework.configuration import build_model_specification

from vivarium_gates_mncnh.constants.data_keys import (
    MATERNAL_HEMORRHAGE,
    MATERNAL_SEPSIS,
    OBSTRUCTED_LABOR,
    PREGNANCY,
)
from vivarium_gates_mncnh.constants.data_values import (
    COLUMNS,
    HEMORRHAGE_SEVERITY,
    PIPELINES,
    PREGNANCY_OUTCOMES,
    SIMULATION_EVENT_NAMES,
    SIMULATION_STEPS,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [ ]:
# Build a small interactive simulation for quick checks
model_spec_path = Path(vivarium_gates_mncnh.__file__).parent / 'model_specifications/model_spec.yaml'
base_spec = build_model_specification(model_spec_path)

DRAW = int(base_spec.configuration.input_data.input_draw_number)
DRAW_STR = 'draw_' + str(DRAW)
POP_SIZE = 100_000

STEP_MAPPER = {name: i + 1 for i, name in enumerate(SIMULATION_STEPS)}

BIRTH_OUTCOMES = [
    PREGNANCY_OUTCOMES.STILLBIRTH_OUTCOME,
    PREGNANCY_OUTCOMES.LIVE_BIRTH_OUTCOME,
]
INTRAPARTUM_DISORDER_COLUMNS = [
    COLUMNS.OBSTRUCTED_LABOR,
    COLUMNS.POSTPARTUM_HEMORRHAGE,
    COLUMNS.MATERNAL_SEPSIS,
    COLUMNS.RESIDUAL_MATERNAL_DISORDERS,
]
APH_SEVERITY_COL = f'{COLUMNS.ANTEPARTUM_HEMORRHAGE}_severity'
PPH_SEVERITY_COL = f'{COLUMNS.POSTPARTUM_HEMORRHAGE}_severity'

def make_sim(scenario: str = 'baseline', pop_size: int = POP_SIZE) -> InteractiveContext:
    spec = deepcopy(base_spec)
    spec.configuration.population.population_size = pop_size
    spec.configuration.intervention.scenario = scenario
    return InteractiveContext(spec)

def run_to_step(sim: InteractiveContext, step_name: str) -> InteractiveContext:
    sim.take_steps(STEP_MAPPER[step_name])
    return sim

artifact = Artifact(base_spec.configuration.input_data.artifact_path)
print('Using artifact:', base_spec.configuration.input_data.artifact_path)
print('Using draw column:', DRAW_STR)
print('Population size:', POP_SIZE)

## 1) APH incidence-risk distribution & [0, 1] bound (Expectation 1)

In [ ]:
sim_aph = make_sim('baseline')
run_to_step(sim_aph, SIMULATION_EVENT_NAMES.ANTEPARTUM_HEMORRHAGE)

aph_pop = sim_aph.get_population([
    COLUMNS.PREGNANCY_OUTCOME,
    COLUMNS.ANTEPARTUM_HEMORRHAGE,
    COLUMNS.MOTHER_AGE,
])
aph_risk = sim_aph.get_population(PIPELINES.ANTEPARTUM_HEMORRHAGE_INCIDENCE_RISK)
if isinstance(aph_risk, pd.DataFrame):
    aph_risk = aph_risk.iloc[:, 0]
aph_risk = aph_risk.astype(float)

print('APH incidence risk: min={:.5f} max={:.5f} mean={:.5f}'.format(
    aph_risk.min(), aph_risk.max(), aph_risk.mean()))
assert (aph_risk >= 0).all() and (aph_risk <= 1).all(), 'APH incidence risk outside [0, 1]'
assert aph_risk.max() > 0, 'APH incidence risk is identically zero'

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(aph_risk, bins=50)
ax.axvline(0, color='k', linestyle='--')
ax.axvline(1, color='k', linestyle='--')
ax.set_xlabel('antepartum_hemorrhage.incidence_risk')
ax.set_ylabel('simulant count')
ax.set_title('APH incidence-risk distribution (must lie in [0, 1])')
plt.show()

## 2) APH incidence by pregnancy outcome — zero for partial_term (Expectations 1 & 5)

In [ ]:
by_outcome = (
    aph_pop.assign(aph=aph_pop[COLUMNS.ANTEPARTUM_HEMORRHAGE].astype(bool))
    .groupby(COLUMNS.PREGNANCY_OUTCOME)['aph']
    .agg(['sum', 'count', 'mean'])
    .rename(columns={'sum': 'aph_cases', 'count': 'n', 'mean': 'aph_incidence'})
)
display(by_outcome)

partial_term_cases = int(
    by_outcome.loc[PREGNANCY_OUTCOMES.PARTIAL_TERM_OUTCOME, 'aph_cases']
    if PREGNANCY_OUTCOMES.PARTIAL_TERM_OUTCOME in by_outcome.index else 0
)
assert partial_term_cases == 0, f'{partial_term_cases} partial_term simulants have APH == True'
print('PASSED: zero APH cases among partial_term pregnancies')

fig, ax = plt.subplots(figsize=(7, 4))
by_outcome['aph_incidence'].plot(kind='bar', ax=ax)
ax.set_ylabel('APH incidence (cases / n)')
ax.set_title('APH incidence by pregnancy outcome (partial_term must be 0)')
plt.show()

## 3) Intrapartum-disorder incidence-risk denominators (Expectation 3)

`birth_rate = (1 + SBR) * ASFR`; `aph_csmr = csmr_c367 * (1 - postpartum_fraction)`. The intrapartum denominator is `birth_rate - aph_csmr`, so every intrapartum incidence risk is scaled up by `birth_rate / (birth_rate - aph_csmr) > 1` versus dividing by `birth_rate` alone.

In [ ]:
def load_female_by_age(art_key: str) -> pd.DataFrame:
    art = artifact.load(art_key).reset_index()
    if 'sex' in art.columns:
        sub = art[art['sex'] == 'Female']
        if not sub.empty:
            art = sub
    group_cols = [c for c in ['age_start', 'age_end'] if c in art.columns]
    if group_cols:
        art = art.groupby(group_cols, as_index=False)[DRAW_STR].mean()
        return art.sort_values(group_cols).reset_index(drop=True)
    return art[[DRAW_STR]].mean().to_frame().T

asfr = load_female_by_age(PREGNANCY.ASFR).set_index(['age_start', 'age_end'])[DRAW_STR]
sbr = artifact.load(PREGNANCY.SBR)
sbr_val = float(sbr[DRAW_STR].mean()) if hasattr(sbr, 'columns') and DRAW_STR in sbr.columns else float(np.asarray(sbr).mean())
csmr_c367 = load_female_by_age(MATERNAL_HEMORRHAGE.CSMR).set_index(['age_start', 'age_end'])[DRAW_STR]
pp_fraction = load_female_by_age(MATERNAL_HEMORRHAGE.POSTPARTUM_FRACTION).set_index(['age_start', 'age_end'])[DRAW_STR]

birth_rate = (1 + sbr_val) * asfr
aph_csmr = csmr_c367 * (1 - pp_fraction)
denom_bump = birth_rate / (birth_rate - aph_csmr)

denom_tbl = pd.DataFrame({
    'birth_rate': birth_rate,
    'aph_csmr': aph_csmr,
    'birth_rate_minus_aph_csmr': birth_rate - aph_csmr,
    'denominator_bump': denom_bump,
}).dropna()
display(denom_tbl)

assert (denom_tbl['denominator_bump'] >= 1.0).all(), 'Intrapartum denominator bump must be >= 1'
print('PASSED: birth_rate - aph_csmr <= birth_rate, so intrapartum incidence risks are scaled up')

In [ ]:
# For sepsis and obstructed labor, show incidence/birth_rate vs incidence/(birth_rate - aph_csmr).
rows = []
for label, key in [
    ('maternal_sepsis', MATERNAL_SEPSIS.RAW_INCIDENCE_RATE),
    ('obstructed_labor', OBSTRUCTED_LABOR.RAW_INCIDENCE_RATE),
]:
    raw = load_female_by_age(key).set_index(['age_start', 'age_end'])[DRAW_STR]
    per_birth = (raw / birth_rate).dropna()
    per_birth_minus_aph = (raw / (birth_rate - aph_csmr)).dropna()
    rows.append({
        'disorder': label,
        'mean_risk_over_birth_rate': float(per_birth.mean()),
        'mean_risk_over_birth_rate_minus_aph_csmr': float(per_birth_minus_aph.mean()),
    })
denom_compare = pd.DataFrame(rows).set_index('disorder')
denom_compare['ratio'] = (
    denom_compare['mean_risk_over_birth_rate_minus_aph_csmr']
    / denom_compare['mean_risk_over_birth_rate']
)
display(denom_compare)
assert (denom_compare['ratio'] >= 1.0).all(), 'Intrapartum incidence risk must be larger with the new denominator'

fig, ax = plt.subplots(figsize=(7, 4))
denom_compare[['mean_risk_over_birth_rate', 'mean_risk_over_birth_rate_minus_aph_csmr']].plot(kind='bar', ax=ax)
ax.set_ylabel('mean incidence risk')
ax.set_title('Intrapartum incidence risk: /birth_rate vs /(birth_rate - aph_csmr)')
plt.show()

## 4) APH per-birth vs per-pregnancy (Expectation 2)

APH risk = `(1 - postpartum_fraction) * incidence_c367 / denominator`. Moving from a per-pregnancy denominator (which included abortion/miscarriage c995 + ectopic c374 incidence) to a per-birth denominator (`birth_rate`) drops those terms, so per-birth APH risk is larger. Below, the per-pregnancy proxy adds the AME (c995 + c374) incidence back into the denominator to show the direction.

In [ ]:
inc_c367 = load_female_by_age(MATERNAL_HEMORRHAGE.RAW_INCIDENCE_RATE).set_index(['age_start', 'age_end'])[DRAW_STR]
miscarriage = load_female_by_age(PREGNANCY.RAW_INCIDENCE_RATE_MISCARRIAGE).set_index(['age_start', 'age_end'])[DRAW_STR]
ectopic = load_female_by_age(PREGNANCY.RAW_INCIDENCE_RATE_ECTOPIC).set_index(['age_start', 'age_end'])[DRAW_STR]
ame_incidence = (miscarriage + ectopic)

aph_numerator = (1 - pp_fraction) * inc_c367
aph_per_birth = (aph_numerator / birth_rate)
aph_per_pregnancy_proxy = (aph_numerator / (birth_rate + ame_incidence))

aph_denom_tbl = pd.DataFrame({
    'aph_per_birth': aph_per_birth,
    'aph_per_pregnancy_proxy': aph_per_pregnancy_proxy,
}).dropna()
aph_denom_tbl['per_birth_over_per_pregnancy'] = (
    aph_denom_tbl['aph_per_birth'] / aph_denom_tbl['aph_per_pregnancy_proxy']
)
display(aph_denom_tbl)
assert (aph_denom_tbl['aph_per_birth'] >= aph_denom_tbl['aph_per_pregnancy_proxy']).all(), (
    'APH per-birth risk must be >= per-pregnancy proxy'
)
print('PASSED: APH per-birth risk >= per-pregnancy proxy for all age groups')

fig, ax = plt.subplots(figsize=(8, 4))
aph_denom_tbl[['aph_per_birth', 'aph_per_pregnancy_proxy']].reset_index(drop=True).plot(ax=ax, marker='o')
ax.set_xlabel('age group index')
ax.set_ylabel('APH incidence risk')
ax.set_title('APH per-birth vs per-pregnancy proxy by age group')
plt.show()

## 5) Cause-of-death breakdown — APH vs intrapartum deaths are disjoint (Expectations 4 & 6)

In [ ]:
sim_mort = make_sim('baseline')
run_to_step(sim_mort, SIMULATION_EVENT_NAMES.MORTALITY)

mort_pop = sim_mort.get_population([
    COLUMNS.MOTHER_ALIVE,
    COLUMNS.MOTHER_CAUSE_OF_DEATH,
    COLUMNS.PREGNANCY_OUTCOME,
    COLUMNS.ABORTION_MISCARRIAGE_ECTOPIC_PREGNANCY,
    APH_SEVERITY_COL,
    PPH_SEVERITY_COL,
] + INTRAPARTUM_DISORDER_COLUMNS)

dead = mort_pop.loc[~mort_pop[COLUMNS.MOTHER_ALIVE]]
cod_breakdown = dead[COLUMNS.MOTHER_CAUSE_OF_DEATH].value_counts()
display(cod_breakdown.to_frame('deaths'))

fig, ax = plt.subplots(figsize=(8, 4))
cod_breakdown.plot(kind='bar', ax=ax)
ax.set_ylabel('mother deaths')
ax.set_title('Maternal cause-of-death breakdown')
plt.tight_layout()
plt.show()

In [ ]:
# Expectation 4: APH death excludes any intrapartum disorder / intrapartum death
aph_dead = mort_pop.index[mort_pop[COLUMNS.MOTHER_CAUSE_OF_DEATH] == COLUMNS.ANTEPARTUM_HEMORRHAGE]
intrapartum_dead = mort_pop.index[mort_pop[COLUMNS.MOTHER_CAUSE_OF_DEATH].isin(INTRAPARTUM_DISORDER_COLUMNS)]
print(f'APH deaths: {len(aph_dead)}; intrapartum-disorder deaths: {len(intrapartum_dead)}')

overlap = aph_dead.intersection(intrapartum_dead)
assert len(overlap) == 0, f'{len(overlap)} simulants recorded both APH and intrapartum death'

gating = {}
for disorder in INTRAPARTUM_DISORDER_COLUMNS:
    gating[disorder] = int(mort_pop.loc[aph_dead, disorder].astype(bool).sum())
gating_tbl = pd.Series(gating, name='intrapartum_flags_among_aph_dead').to_frame()
display(gating_tbl)
assert gating_tbl['intrapartum_flags_among_aph_dead'].sum() == 0, (
    'APH-dead simulants must not carry any intrapartum disorder incidence'
)
print('PASSED: APH deaths are disjoint from intrapartum disorders and intrapartum deaths')

In [ ]:
# Expectation 5 (AME side) and Expectation 6 (severe-only hemorrhage deaths)
ame_non_partial = int((
    mort_pop[COLUMNS.ABORTION_MISCARRIAGE_ECTOPIC_PREGNANCY].astype(bool)
    & (mort_pop[COLUMNS.PREGNANCY_OUTCOME] != PREGNANCY_OUTCOMES.PARTIAL_TERM_OUTCOME)
).sum())
assert ame_non_partial == 0, f'{ame_non_partial} non-partial_term simulants have AME == True'
print('PASSED: AME assigned only to partial_term pregnancies')

for cause, sev_col in [
    (COLUMNS.ANTEPARTUM_HEMORRHAGE, APH_SEVERITY_COL),
    (COLUMNS.POSTPARTUM_HEMORRHAGE, PPH_SEVERITY_COL),
]:
    died = mort_pop.loc[mort_pop[COLUMNS.MOTHER_CAUSE_OF_DEATH] == cause]
    non_severe = died.loc[died[sev_col] != HEMORRHAGE_SEVERITY.SEVERE]
    assert len(non_severe) == 0, f'{len(non_severe)} non-severe {cause} deaths found'
    print(f'PASSED: all {len(died)} {cause} deaths were severe')